# GPT Importance Definition Analysis

## File generation

In [ ]:
import pickle
import os
import json
import pandas as pd

with open(os.path.join("info.json"), "r") as file:
    info = json.load(file)

importance_prompt = "Please define 'importance'. You will use this definition to calculate token importance when prompted later."
revise_prompt = "Based on the results of all previous masking iterations, adjust the definition of 'importance' for yourself as you see fit."

for directory, column in [("gpt_index_masking", "gpt_index_value"), ("gpt_index_word_masking", "gpt_index_word_value")]:
    output = []
    for id in info['id']:
        output.append({'id': id})
        
        with open(os.path.join(directory, 'messages', f'{id}.pkl'), "rb") as file:
            messages = pickle.load(file)
            
            for i, m in enumerate(messages):
                if type(m) == dict:
                    if m['content'] == importance_prompt:
                        output[-1][f"{column}_definition"] = messages[i+1].content
                    elif m['content'] == revise_prompt:
                        output[-1][f"{column}_revise"] = messages[i+1].content

    df_output = pd.DataFrame(output)
    display(df_output)
    df_output.to_csv(os.path.join('results', f"importance_{directory}.csv"), index=False)

## Analysis

In [ ]:
import pandas as pd

files = [
    'importance_gpt_index_masking.xlsx',
    'importance_gpt_index_word_masking.xlsx'
]

for file in files:
    df = pd.read_excel(os.path.join('results', file))
    display(df.describe())

,id,initial_change_in_probability,initial_change_in_logits,initial_description,revised_probability,revised_logits,revised_average,revised_cumulative,revised_normalized_by_original_probability,revised_weighted_by_magnitude_of_change,revised_normalized_by_number_of_tokens
count,2.000000e+02,200.0,0.0,0.0,199.0,4.0,196.0,0.0,37.0,59.0,16.0
mean,2.745739e+07,1.0,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,1.0
std,1.018589e+07,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0
min,1.063280e+07,1.0,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,1.0
25%,1.916052e+07,1.0,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,1.0
50%,3.081323e+07,1.0,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,1.0
75%,3.610730e+07,1.0,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,1.0
max,3.881131e+07,1.0,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,1.0


,id,initial_change_in_probability,initial_change_in_logits,initial_description,revised_probability,revised_logits,revised_average,revised_cumulative,revised_normalized_by_original_probability,revised_weighted_by_magnitude_of_change,revised_normalized_by_number_of_tokens,revised_masking_frequency,revised_context,revised_consistency
count,2.000000e+02,200.0,0.0,0.0,200.0,0.0,200.0,0.0,67.0,6.0,9.0,16.0,15.0,17.0
mean,2.745739e+07,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0
std,1.018589e+07,0.0,NaN,NaN,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
min,1.063280e+07,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0
25%,1.916052e+07,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0
50%,3.081323e+07,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0
75%,3.610730e+07,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0
max,3.881131e+07,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0


#### Message Analysis

In [ ]:
import joblib
import pickle
import os
from pathlib import Path

path = Path("gpt_index_word_masking/messages/10632803.pkl")
message = joblib.load(path)

for m in message:
    print(m)

{'role': 'developer', 'content': "\n\nYou are a machine learning model explainer.\n \nYou are tasked with explaining binary text classification encoder-only transformer model's predictions by perturbing its input tokens, similar to perturbation based XAI frameworks like LIME or SHAP.\n\n\nYou will be provided the number of tokens, the input tokens, the logits for both positive and negative classes, the probability for the positive class. You will also be provided the manual appraisal criteria (appraised using the full text).\n\n\nFor a given instance, determine which tokens have the greatest impact on the model's prediction by systematically masking them. You will:\n\n1. Receive an instance with the number of tokens, the tokens, and model outputs without any masking.\n2. Define 'importance' for yourself. The importance should be a float. A negative and positive float should indicate that the token increases the chance of a negative and positive classification, respectively.\n3. Generat